# AniFlow — Colab Edition
نسخة تشغيل AniFlow على Google Colab. شغّل الخلايا بالترتيب. يفضّل Runtime من نوع GPU.

- Vision: `qwen2.5vl:3b`
- Script writer: `qwen2.5:3b`
- TTS: Edge TTS
- Video: FFmpeg

يدعم إدخال روابط المنصات التي يستطيع مستخرج AniFlow قراءتها، أو رفع مجلد صور / ZIP من واجهة Source. ليس مضمونًا أن تعمل كل منصة؛ المواقع التي تتطلب تسجيل دخول أو تمنع الوصول قد تحتاج مستخرجًا مخصصًا.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_HOME = Path('/content/drive/MyDrive/AniFlow')
DRIVE_HOME.mkdir(parents=True, exist_ok=True)
print('Persistent workspace:', DRIVE_HOME)

In [ ]:
# System tools + project checkout
!apt-get update -qq
!apt-get install -y -qq ffmpeg chromium git curl
!git clone https://github.com/sef014010-lgtm/Aniflow.git /content/AniFlow
%cd /content/AniFlow
!python -m pip install --upgrade pip
!pip install -r requirements.txt
!python -m compileall -q app.py config.py pipeline
print('AniFlow source and Python syntax check ready.')

In [ ]:
# Keep projects, inputs, assets, downloads and generated media on Google Drive
import shutil
from pathlib import Path
PROJECT = Path('/content/AniFlow')
persistent = ['projects','casts','assets','input','output','manhwa_download','manhwa_stitch','manhwa_detect','manhwa_crop','manhwa_text','manhwa_audio','manhwa_clips']
for name in persistent:
    src = PROJECT / name
    dst = DRIVE_HOME / name
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists() and src.exists() and not src.is_symlink():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    if src.is_symlink() or src.exists():
        if src.is_symlink() or src.is_file(): src.unlink()
        else: shutil.rmtree(src)
    dst.mkdir(parents=True, exist_ok=True)
    src.symlink_to(dst, target_is_directory=True)
print('Persistent folders linked to Google Drive.')

In [ ]:
# Install Ollama and pull the two requested small models
# Colab images may not include zstd, which the Ollama installer requires.
!apt-get update -qq
!apt-get install -y -qq zstd curl
!command -v zstd
!curl -fsSL https://ollama.com/install.sh | sh
!command -v ollama
import os, subprocess, time, socket
env = os.environ.copy()
env['OLLAMA_HOST'] = '127.0.0.1:11434'
ollama_log = open('/content/ollama.log', 'w')
ollama_proc = subprocess.Popen(['ollama','serve'], env=env, stdout=ollama_log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        with socket.create_connection(('127.0.0.1',11434), timeout=1): break
    except OSError: time.sleep(2)
else: raise RuntimeError('Ollama failed to start. See /content/ollama.log')
!ollama pull qwen2.5vl:3b
!ollama pull qwen2.5:3b
os.environ['OLLAMA_HOST'] = 'http://127.0.0.1:11434'
os.environ['OLLAMA_MODEL'] = 'qwen2.5vl:3b'
os.environ['VISION_MODEL'] = 'qwen2.5vl:3b'
os.environ['WRITER_MODEL'] = 'qwen2.5:3b'
os.environ['TEXT_WRITER_MODEL'] = 'qwen2.5:3b'
os.environ['COLAB_RELEASE_TAG'] = os.getenv('COLAB_RELEASE_TAG','colab')
print('Ollama models are ready.')

In [ ]:
# Start the Flask / Socket.IO application and expose it through Colab
import requests, subprocess, time, os
app_env = os.environ.copy()
app_log = open('/content/aniflow.log','w')
app_proc = subprocess.Popen(['python','app.py'], cwd='/content/AniFlow', env=app_env, stdout=app_log, stderr=subprocess.STDOUT)
for attempt in range(90):
    try:
        response = requests.get('http://127.0.0.1:8080/api/status', timeout=5)
        if response.ok: break
    except requests.RequestException: pass
    time.sleep(2)
else: raise RuntimeError('AniFlow did not start. Inspect /content/aniflow.log')
from google.colab import output
APP_URL = output.eval_js("google.colab.kernel.proxyPort(8080)")
print('AniFlow is running:', APP_URL)
print('Open the URL above. In Source, paste a supported chapter/series URL or upload a folder / ZIP.')

In [ ]:
# Optional: check Ollama and the app health
import requests, subprocess
print('App status:', requests.get('http://127.0.0.1:8080/api/status', timeout=15).json())
print('Ollama tags:', requests.get('http://127.0.0.1:11434/api/tags', timeout=15).json().get('models', []))

## Notes
- Upload a directory using **Upload image folder** or a ZIP using **Upload ZIP** in the Source screen. The server stages images into the project workspace; no desktop Tkinter picker is used in Colab.
- You can also enter a path under `/content/drive/MyDrive/AniFlow/...` in the Chapter Folder Path field.
- Output folders are linked to Google Drive. Colab GPU and session availability depend on Google.
- Existing scraper strategies are reused and broadened. Some services use authentication, DRM, signed APIs or dynamic readers and may still need a dedicated extractor. Use only content you are authorized to access.
- If a cell fails, check `/content/aniflow.log` or `/content/ollama.log` before rerunning.